# Defect detection pipeline

Does super-resolving a low-resolution capture improve automated defect detection? Every method reconstructs the same LR test split at the HR frame size, and every row is classified with the one fine-tuned VGG16.

Thirteen rows are compared, and every figure reads them in the same order: the LR baseline, the four interpolations, the four advanced classic algorithms, the three learned models (SRCNN, EDSR, ESRGAN) and the HR ceiling. The twelve resampled rows all produce RGB images at 478x478, which is the regime the classifier was trained on.

The **LR row is the baseline rescaled to the reference frame with nearest-neighbour interpolation**, and has to be described as such. Replicating pixels adds no detail, so the row still measures the low-resolution capture, but it puts the baseline at the same frame size and the same 81 voting patches as every other row. Scored at its native 239x239 it aggregated only 16 patches and showed the object at twice the apparent scale the classifier was trained on, so part of any gap in favour of super-resolution came from the protocol rather than from the resolution.

A single classifier scores every row, references included, which is what makes the comparison readable: a difference between two rows is a difference between the images, never between two models fitted separately. The two references sit at the ends of every figure and bracket the reconstructions, and the ceiling is what turns an absolute accuracy into a readable one: a method at 91 % means something different if HR reaches 93 % than if it reaches 99 %.

Requires `4_srcnn`, `5_edsr`, `6_esrgan` and `8_vgg16` to have been run.

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from srlib.constants import CLASS_LABELS_PATH, HR_ROOT, LR_ROOT
from srlib.dataset.loading import load_defect_detection_pipeline_dataset
from srlib.defect_detection.pipeline import DefectDetectionPipeline
from srlib.model_registry import print_available_runs

c:\Users\bgmanuel\anaconda3\envs\py310\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\bgmanuel\anaconda3\envs\py310\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.

## Load the test split

The same split the classifier was trained against, so the test images were never seen during training.

In [2]:
(
    X_LR_train, X_HR_train, y_train,
    X_LR_val, X_HR_val, y_val,
    X_LR_test, X_HR_test, y_test,
) = load_defect_detection_pipeline_dataset(HR_ROOT, LR_ROOT, CLASS_LABELS_PATH)

print(f"LR splits -> train: {X_LR_train.shape}, val: {X_LR_val.shape}, test: {X_LR_test.shape}")
print(f"HR splits -> train: {X_HR_train.shape}, val: {X_HR_val.shape}, test: {X_HR_test.shape}")
print(f"y splits  -> train: {y_train.shape}, val: {y_val.shape}, test: {y_test.shape}")


 Detection pipeline dataset | full frames
  images    798 indexed -> 798 kept -> 574 train / 64 val / 160 test


  frames    LR 239x239 | HR 478x478
  test      160 images, the split the pipeline scores
  done in 6.0s

LR splits -> train: (574, 239, 239, 3), val: (64, 239, 239, 3), test: (160, 239, 239, 3)
HR splits -> train: (574, 478, 478, 3), val: (64, 478, 478, 3), test: (160, 478, 478, 3)
y splits  -> train: (574,), val: (64,), test: (160,)


## Run the pipeline

Leave a run as `None` to take the most recent checkpoint of that model, or pin a timestamp to reproduce an older comparison.

In [3]:
print_available_runs()

SRCNN: 1 run(s)
  - 20260914_180459 (latest)
EDSR: 1 run(s)
  - 20260914_204828 (latest)
ESRGAN: 1 run(s)
  - 20260915_175433 (latest)
VGG16: 1 run(s)
  - 20260914_235902 (latest)


In [4]:
pipeline = DefectDetectionPipeline(
    X_LR_test=X_LR_test,
    X_HR_test=X_HR_test,
    y_test=y_test,
    srcnn_run=None,
    edsr_run=None,
    esrgan_run=None,
    vgg16_run=None,
)

sr_images, labels, confidences = pipeline.run()

Resolved runs
  SRCNN   20260914_180459
  EDSR    20260914_204828
  ESRGAN  20260915_175433
  VGG16   20260914_235902
  HR frame size: 478 x 478
Loading models
Loaded pretrained model from c:\Users\bgmanuel\MasterInteligenciaArtificial\Periodo2\TFM\Super-Resolution-Images-for-3D-Printing-Defect-Detection\models\SRCNN\SRCNN_20260914_180459\SRCNN_20260914_180459.h5
Loaded pretrained model from c:\Users\bgmanuel\MasterInteligenciaArtificial\Periodo2\TFM\Super-Resolution-Images-for-3D-Printing-Defect-Detection\models\EDSR\EDSR_20260914_204828\EDSR_x2_20260914_204828.h5
- Generator loaded from: c:\Users\bgmanuel\MasterInteligenciaArtificial\Periodo2\TFM\Super-Resolution-Images-for-3D-Printing-Defect-Detection\models\ESRGAN\ESRGAN_20260915_175433\ESRGAN_generator_x2_20260915_175433.h5
- Discriminator loaded from: c:\Users\bgmanuel\MasterInteligenciaArtificial\Periodo2\TFM\Super-Resolution-Images-for-3D-Printing-Defect-Detection\models\ESRGAN\ESRGAN_20260915_175433\ESRGAN_discriminator_x2_202

  LR (nearest upscale)      0.07s


  SRCNN                    16.14s


  EDSR                     14.43s


  ESRGAN                   50.92s


...

  Freq-extrapolation       15.45s  accuracy=0.9500


  SRCNN                    14.15s  accuracy=0.9000


  EDSR                     17.08s  accuracy=0.9125


  ESRGAN                   16.67s  accuracy=0.9875


  HR                       16.16s  accuracy=1.0000
Predictions done in 206.07s

Pipeline finished in 324.03s


## Confusion matrices

One matrix per method, in reading order: the LR baseline, the classic families, the learned models and the HR ceiling.

In [ ]:
pipeline.plot_confusion_matrices()

## Classification report

Accuracy, macro recall and both F1 aggregates, plus the same scores resolved per class. The dataset is imbalanced, so macro F1 is the figure to read rather than accuracy.

In [ ]:
fig, axes, report_metrics = pipeline.plot_classification_reports()

## Confidence

Global mean confidence, the same split into correct and wrong predictions, and the error rate. The three together are what tell whether a method is confidently wrong.

In [ ]:
fig, axes, confidence_metrics = pipeline.plot_confidence()

## Qualitative comparison

Every reconstruction of a single test image with the prediction it produced. Change `index` to inspect another sample.

In [ ]:
pipeline.plot_sample_predictions(index=9)

Time and memory of the learned models are reported in notebook 7, measured over training and over the test evaluation. Reconstructing a frame patch by patch is a cost of this pipeline, not of the models, so it is not attributed to them.